# Part 3.1: Feature Selection

In [18]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd 
from lightgbm import LGBMClassifier

In [14]:
import mlflow
import sys
sys.path.append(r'C:\Users\Hzaab\Desktop\MLSD project\Part_4')
from mlflowFunction import log_model_run

mlflow.set_tracking_uri("http://127.0.0.1:5000")
print("Tracking URI:", mlflow.get_tracking_uri())
mlflow.set_experiment("Model Development")
mlflow.utils.logging_utils.disable_logging()


Tracking URI: http://127.0.0.1:5000


In [24]:
train_processed = pd.read_parquet('C:\\Users\\Hzaab\\Desktop\\MLSD project\\data\\preprocessed\\train_processed.parquet', engine='fastparquet')
# Logistic Regression model evaluation function
def evaluate_model(X, y, name, model):
    scores = cross_validate(model, X, y, cv=5, scoring=["accuracy", "precision", "recall", "f1"])
    results = {
        "accuracy": np.mean(scores["test_accuracy"]),
        "f1": np.mean(scores["test_f1"]),
        "recall": np.mean(scores["test_recall"]),
        "precision": np.mean(scores["test_precision"])
    }
    print(f"{name}")
    print(f"F1: {np.mean(scores['test_f1']):.4f}, Precision: {np.mean(scores['test_precision']):.4f}, Recall: {np.mean(scores['test_recall']):.4f}") 
    return results

In [25]:
#Feature Selection and Model Evaluation 
target_col = "fake"

X_all = train_processed.drop(columns=[target_col])
y = train_processed[target_col]

important_features = ["#posts", "#followers", "#follows", "description length"]

X_selected = train_processed[important_features]

model = LogisticRegression(max_iter=1000)
results = evaluate_model(X_all, y, "Logistic Regression | All Features", model)
log_model_run("Logistic Regression | All Features", model, results, X_all, y)

lgbm = LGBMClassifier(random_state=10)
results = evaluate_model(X_all, y, "LightGBM | All Features", lgbm)
log_model_run("LightGBM | All Features", lgbm, results, X_all, y)

log_reg = LogisticRegression(max_iter=1000)
results = evaluate_model(X_selected, y, "Logistic Regression | Important Features", log_reg)
log_model_run("Logistic Regression | Important Features", log_reg, results, X_selected, y)

lgbm = LGBMClassifier(random_state=10)
results = evaluate_model(X_selected, y, "LightGBM | Important Features", lgbm)
log_model_run("LightGBM | Important Features", lgbm, results, X_selected, y)

X_weighted = X_all.copy()
for col in important_features:
    X_weighted[col] = X_weighted[col] * 2

log_reg = LogisticRegression(max_iter=1000)
results = evaluate_model(X_weighted, y, "Logistic Regression | Weighted Important Features", log_reg)
log_model_run("Logistic Regression | Weighted Important Features", log_reg, results, X_weighted, y)
lgbm = LGBMClassifier(random_state=10)
evaluate_model(X_weighted, y, "LightGBM | Weighted Important Features", lgbm)

# Try different C values
for c in [0.1, 0.5, 1, 5, 10]:
    model = LogisticRegression(C=c, max_iter=1000)
    evaluate_model(X_all, y, f"Logistic Regression | All Features | C={c}", model)

    scores = cross_validate(model, X_all, y, cv=5, scoring=["f1", "precision", "recall"])
    
    print(f"\nC = {c}")
    print("F1 Score  :", scores["test_f1"].mean())
    print("Precision :", scores["test_precision"].mean())
    print("Recall    :", scores["test_recall"].mean())




Logistic Regression | All Features
F1: 0.9549, Precision: 0.9818, Recall: 0.9300
🏃 View run Logistic Regression | All Features at: http://127.0.0.1:5000/#/experiments/2/runs/8bea60c6f46a4b17b8edeabbe69e2174
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 320, number of negative: 1280
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000124 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1229
[LightGBM] [Info] Number of data points in the train set: 1600, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.200000 -> initscore=-1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [W

In [26]:
# Add features
X_new = train_processed.copy()

# New features
X_new["engagement_ratio"] = X_new["#followers"] / (X_new["#follows"] + 1e-6)
X_new["activity_rate"] = X_new["#posts"] / (X_new["#followers"] + 1e-6)
X_new["follow_ratio"] = X_new["#follows"] / (X_new["#followers"] + 1e-6)

# Clean any bad values
X_new = X_new.replace([np.inf, -np.inf], 0)
X_new = X_new.fillna(0)


# Important + new only
selected_plus_new = important_features + [
    "engagement_ratio",
    "activity_rate",
    "follow_ratio"
]


model = LogisticRegression(max_iter=1000)
results = evaluate_model(X_new[selected_plus_new], y, "Logistic Regression | Important + New Features Only", model)
log_model_run("Logistic Regression | Important + New Features Only", model, results, X_new[selected_plus_new], y)

model = LGBMClassifier(random_state=10)
results = evaluate_model(X_new[selected_plus_new], y, "LightGBM | Important + New Features Only", model)
log_model_run("LightGBM | Important + New Features Only", model, results, X_new[selected_plus_new], y)



Logistic Regression | Important + New Features Only
F1: 0.8379, Precision: 0.8863, Recall: 0.7950
🏃 View run Logistic Regression | Important + New Features Only at: http://127.0.0.1:5000/#/experiments/2/runs/032bb32443ec474bbf6a1fe655b493f8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 320, number of negative: 1280
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000100 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1651
[LightGBM] [Info] Number of data points in the train set: 1600, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.200000 -> initscore=-1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 320, number 

### C5 or C10 seems to give the best balance of precision and recall.